In [ ]:
from pathlib import Path
from typing import Annotated, Literal
from datetime import datetime
from copy import copy

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib.colors import PowerNorm
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
import matplotlib as mpl
import seaborn as sns

import geopandas
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs
import matplotlib as mpl
from mpl_sankey import sankey

from IPython.display import display, HTML, Markdown

from climate_health_map.shared import get_logger
from climate_health_map.data.labels import LABELS, Collection, Group, AggTopic, AggAggTopic
from climate_health_map.data.geographies import load_country_infos, load_grid_data, load_annual_population, merge_grid_info, LOCATION_FEATURE_CODES
from climate_health_map.data.export.lancet.loaders import read_base_data
from climate_health_map.data.export.utils import label_group_counts
from climate_health_map.data.export.lancet.regional_groups import region_groups

logger = get_logger('notebook', loglevel='INFO')

In [ ]:
OUTPUT_DIR = Path('../data/exports/lancet/LatinAmerica/Figures')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Load data

In [ ]:
try:
    shapes = geopandas.read_file('../data/shapes_2026/WB_GAD_ADM0_complete.shp', encoding='utf-8')
except:
    logger.warning('Official World Bank shapefile missing, falling back to natural_earth default')
    shpfilename = shpreader.natural_earth(
        resolution='10m',
        category='cultural',
        name='admin_0_countries'
    )
    shapes = geopandas.read_file(shpfilename, encoding='utf-8')

# See https://cartopy.readthedocs.io/stable/reference/projections.html

PROJECTION = ccrs.PlateCarree  
AXIS_PROJECTION = ccrs.EqualEarth # Lancert author guideline wants Equal Earth
# #EckertIII

shapes_reg = shapes[shapes['ISO_A3'].isin(REGION['Countries'])]
EXTENT = np.array([
        max(-180, np.array([poly.bounds for poly in shapes_reg['geometry']])[:,[0,2]].min()),
        min(180, np.array([poly.bounds for poly in shapes_reg['geometry']])[:,[0,2]].max()),
        max(-90, np.array([poly.bounds for poly in shapes_reg['geometry']])[:,[1,3]].min()),
        min(90, np.array([poly.bounds for poly in shapes_reg['geometry']])[:,[1,3]].max()),
], dtype=float)
print(EXTENT)
EXTENT = [-110, -30, -55, 27]

In [ ]:
df, df_locations, df_locations_flat, location_groups, df_affiliations, df_affiliations_flat, affiliation_groups = read_base_data(
    source=Path('../data/exports/2026'),
    year_start=2006,
    year_end=2025,
    filter_mai=True,
    filter_rel=True,
    prefilter_affiliations=True,
    prefilter_locations=True,
    fix_locations=True,
    logger=logger.getChild('reader'),
)
del df_affiliations_flat, df_locations_flat

In [ ]:
df_countries = load_country_infos()
df_grid = load_grid_data()
df_population = load_annual_population()
df_population = df_population[(df_population['year'] >= df['Publication year'].min()) & (df_population['year'] <= df['Publication year'].max())]

In [ ]:
for key in [
    'driver', 
    'event', 
    'health', 
    'expose',
    'attr',
    #'sector', 
    'keywords',
   #'topic-agg',
    'topic-agg-agg',
]:
    for l in LABELS[key].labels:
        #print(f"'{l.column}',  # {l.name}")
        pass

# Some labels are pretty rare. Better drop them and not show it at all.
df.drop(columns=[
    'driver|0',  # CO2rise
    'driver|2',  # Seasonal Change
    'event|0',  # Floods
    'event|3',  # Extreme cold
    'event|4',  # Storms
    'event|5',  # Droughts
    'health|9',  # Renal system
    'health|10',  # Metabolic Disorders
    'attr|1',  # Trend attribution
    'topic-agg-agg|4',  # Other
], inplace=True)

In [ ]:
# Match grid info to locations and items
# This is a bit more complex, because some locations need to be projected to a shape (e.g. country) and then back to grid cells
df, df_locations, grid_counts = merge_grid_info(
    df=df,
    df_locations=df_locations,
    df_grid=df_grid,
    df_population=df_population,
    shapes=shapes,
    location_codes_abstracted=LOCATION_FEATURE_CODES[0],
    location_codes_direct=LOCATION_FEATURE_CODES[3] | LOCATION_FEATURE_CODES[2] | LOCATION_FEATURE_CODES[1],
    logger=logger,
    reference_year=2024,  # We propbably want to use the latest population count for normalisation
    resolution=2.5,
)

df_locations = df_locations.set_index('item_id', drop=False)

In [ ]:
def label_group_counts(
    df: pd.DataFrame,
    group: Group | AggTopic | AggAggTopic,
    geography_filter: Literal['affiliation', 'location', 'region_affiliation', 'region_location'] | None = None,
    count_primary_class: bool = False,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    table_index = pd.RangeIndex(start=df['Publication year'].min(), stop=df['Publication year'].max() + 1, step=1, name='Publication year')

    columns = [label.column for label in group.labels if label.column in df.columns]
    mask = (df[columns].notna() & (df[columns] > threshold)).any(axis=1)
    primary = df[mask][columns].fillna(0).idxmax(axis=1)

    if group.collection == Collection.IMPACTS:
        mask &= df['incl_impacts']
    if geography_filter == 'location':
        mask &= df['incl_location']
    if geography_filter == 'affiliation':
        mask &= df['incl_affiliation']
    if geography_filter == 'region_location':
        mask &= df['incl_region_location']
    if geography_filter == 'region_affiliation':
        mask &= df['incl_region_affiliation']

    data = {}
    totals = {}
    for label in group.labels:
        if label.column not in df.columns:
            continue

        extra_mask = (primary == label.column) if count_primary_class else df[label.column] > threshold

        totals[label.name] = df[mask & extra_mask]['item_id'].nunique()
        data[label.name] = df[mask & extra_mask].groupby('Publication year')['item_id'].nunique()

    return (
        pd.DataFrame(data, index=table_index).fillna(0).astype(int),
        pd.Series(totals).fillna(0).astype(int),
        pd.Series(df[mask].groupby('Publication year')['item_id'].nunique(), index=table_index, name='Total').fillna(0).astype(int),
    )

# General counts

In [ ]:
REGION_NAME = 'Latin America'
REGION = region_groups[REGION_NAME]

df_locations['incl_region'] = df_locations['iso3'].isin(REGION['Countries'])
df_affiliations['incl_region'] = df_affiliations['iso3'].isin(REGION['Countries'])
df['incl_region_loc'] = df.index.isin(df_locations[df_locations['incl_region']].index)#['item_id'].unique())
df['incl_region_aff'] = df.index.isin(df_affiliations[df_affiliations['incl_region']].index)
df_countries['incl_region'] = df_countries['iso3'].isin(REGION['Countries'])
df_locations['incl_region_loc'] = df_locations['incl_region']
df_affiliations['incl_region_aff'] = df_affiliations['incl_region']
df_locations['incl_region_aff'] = df_locations['item_id'].isin(df_affiliations['item_id'].unique())
df_affiliations['incl_region_loc'] = df_affiliations['item_id'].isin(df_locations['item_id'].unique())

display(Markdown(f'''
**Report for {REGION_NAME}**
```json
{REGION}
```

* Size of base data: {df.shape} / {REGION_NAME} filtered (loc): {df[df['incl_region_loc']].shape} / {REGION_NAME} filtered (aff): {df[df['incl_region_aff']].shape} / uniq: {df['item_id'].nunique():,} / uniq filtered (loc): {df[df['incl_region_loc']]['item_id'].nunique():,} / uniq filtered (aff): {df[df['incl_region_aff']]['item_id'].nunique():,}
* Size of location table: {df_locations.shape} / {REGION_NAME} filtered: {df_locations[df_locations['incl_region']].shape} / uniq: {df_locations['item_id'].nunique():,} / uniq filtered: {df_locations[df_locations['incl_region']]['item_id'].nunique():,}
* Size of affiliations table: {df_affiliations.shape} / {REGION_NAME} filtered: {df_affiliations[df_affiliations['incl_region']].shape} / uniq: {df_affiliations['item_id'].nunique():,} / uniq filtered: {df_affiliations[df_affiliations['incl_region']]['item_id'].nunique():,}
'''))

In [ ]:
df_countries[df_countries['incl_region']]

In [ ]:
for key in ['incl_major', 'incl_impacts', 'incl_location', 'incl_affiliation']:
    print(key, df[key].sum())

# Annual counts

In [ ]:
pd.options.display.max_columns = 650
pd.options.display.max_rows = 40

def barplot(fname: str|None=None):
    GR =  1.618  # golden ratio
    fig_height = 5
    
    #['#fc8d62','#bb22b3','#8da0cb']
    deutera_colours =  ['#666E95', '#FFC13D', '#2F86E5', '#ACA6C3', '#483E3C','#E8C6BA', '#867065', '#005186', '#B0860F', '#FFD9B0']
    prota_colours = ['#7D7C01', '#59709E', '#4C5631', '#3F59E8', '#BDBB64','#35A9E0', '#E8E602', '#C1C1C7', '#0E1079']
    trita_colours = ['#5F727A', '#FFB7C2', '#01919A', '#A8A8B4', '#BDE6F4','#4A3F45', '#B77B87', '#00585C', '#932929']
    colours = deutera_colours
    
    fig, ax = plt.subplots(dpi=150, figsize=(fig_height * GR, fig_height))
    
    df_counts[(REGION_NAME, 'Absolute')].drop(columns=['Total']).plot.bar(stacked=True, ax=ax, legend=False, color=colours)
    ax.set_ylabel('Number of publications')
    ax.set_xlabel('Publication year')
    ax.set_title(f'Estimated number of papers published \non the nexus of climate and health with study location in {REGION_NAME}')
    ax.set_xticks(np.arange(0, np.sum(ax.get_xlim()), 5))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    
    share_of_global = df_counts[(REGION_NAME, 'Absolute')] / df_counts[('Global', 'Absolute')] * 100
    
    ax2 = ax.twinx()
    for i, (label, data) in enumerate(share_of_global.items()):
        ax2.plot(data.index.astype(str), data.values, label=label, color=colours[i])
    
    ax2.set_ylabel('Share of global (%)')
    ax2.set_ylim(0, 50)
    
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Line2D([0], [0], label='Total', color=colours[3]))
    ax.legend(handles=handles)
    
    fig.tight_layout()

    if fname is not None:
        fig.savefig(OUTPUT_DIR / f'{fname}.png')
        fig.savefig(OUTPUT_DIR / f'{fname}.svg')
        fig.savefig(OUTPUT_DIR / f'{fname}.pdf')

    return fig, ax



def lineplot(fname: str|None=None):
    num_rows = len(df_counts.T.groupby(level=0))
    num_cols = len(df_counts.T.groupby(level=1)) - 1

    fig = plt.figure(layout='constrained', figsize=(num_cols*3, num_rows*2), dpi=150)
    subfigs = fig.subfigures(2, 1)
    for subfig, (grouping, data) in zip(subfigs, df_counts.T.groupby(level=0)):
        subfig.suptitle(grouping, fontweight='bold')
        axes = subfig.subplots(1, len(data.groupby(level=1)))

        for ai, (subgroup, subdata) in enumerate(data.groupby(level=1)):
            for col, coldata in subdata.iterrows():
                axes[ai].plot(coldata.index, coldata.fillna(np.nan).values, label=col[-1], ls='-' if col[-1] == 'Total' else '--')
            axes[ai].set_title(subgroup)
            axes[ai].legend()

    #fig.tight_layout()

    if fname is not None:
        fig.savefig(OUTPUT_DIR / f'{fname}.png')
        fig.savefig(OUTPUT_DIR / f'{fname}.svg')
        fig.savefig(OUTPUT_DIR / f'{fname}.pdf')
    return fig



def annual_counts_info(count_primary_class: bool, region_filter: str, label_group: str, mask: pd.Series):
    display(Markdown('**Global totals**'))
    annual_counts, totals, annual_totals = label_group_counts(
        df[mask], 
        group=LABELS[label_group],
        geography_filter=None,
        count_primary_class=count_primary_class,
    )
    
    joined = annual_counts.join(annual_totals)
    cols = joined.columns
    share = (joined.T / joined['Total'] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual proportion'], cols]), axis=1)
    cumsum = joined.cumsum().set_axis(pd.MultiIndex.from_product([['CumSum'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    pct = (joined.cumsum().pct_change(axis=0)*100).set_axis(pd.MultiIndex.from_product([['Annual growth in %'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    cagr = ((((joined.cumsum().pct_change(5) + 1) ** (1/5)) - 1) * 100).set_axis(pd.MultiIndex.from_product([['5yr CAGR'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    df_global = joined.set_axis(pd.MultiIndex.from_product([['Absolute'], cols]), axis=1).join(cumsum).join(pct).join(cagr).join(share)

    df_global_total = pd.DataFrame({
        'Absolute count (global)': df_global['Absolute'].sum(axis=0),
        'Share of total (2006–2015)': df_global.loc[2006:2015]['Absolute'].sum(axis=0) / df_global.loc[2006:2015][('Absolute', 'Total')].sum() * 100,
        'Share of total (2016–2025)': df_global.loc[2016:2025]['Absolute'].sum(axis=0) / df_global.loc[2016:2025][('Absolute', 'Total')].sum() * 100,
        'Share of total (2006–2025)': df_global.loc[2006:2025]['Absolute'].sum(axis=0) / df_global.loc[2006:2025][('Absolute', 'Total')].sum() * 100,
        'Growth 2006–2015': ((cumsum.loc[2015] - cumsum.loc[2006]) / cumsum.loc[2006] * 100)['CumSum'],
        'Growth 2016–2025': ((cumsum.loc[2025] - cumsum.loc[2016]) / cumsum.loc[2016] * 100)['CumSum'],
    }).T

    display(df_global_total)
    
    display(Markdown(f'**{REGION_NAME} totals**'))
    print(df[mask & df[region_filter]].shape)
    annual_counts, totals, annual_totals = label_group_counts(
        df[mask & df[region_filter]], 
        group=LABELS[label_group],
        geography_filter=region_filter,
        count_primary_class=count_primary_class,
    )
    
    joined = annual_counts.join(annual_totals)
    cols = joined.columns

    share = (joined.T / joined['Total'] * 100).T.set_axis(pd.MultiIndex.from_product([[f'Annual proportion of {REGION_NAME}'], cols]), axis=1)
    share_global = (joined.T / df_global[('Absolute', 'Total')] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual proportion of Global'], cols]), axis=1)
    cumsum = joined.cumsum().set_axis(pd.MultiIndex.from_product([['CumSum'], cols]), axis=1).astype('Int32').replace({np.inf:pd.NA, np.nan: pd.NA})
    pct = (joined.cumsum().pct_change(axis=0)*100).set_axis(pd.MultiIndex.from_product([['Annual growth in %'], cols]), axis=1).astype('Float64').replace({np.inf:pd.NA, np.nan: pd.NA})
    cagr = ((((joined.cumsum().pct_change(5) + 1) ** (1/5)) - 1) * 100).set_axis(pd.MultiIndex.from_product([['5yr CAGR'], cols]), axis=1).astype('Float64').replace({np.inf:pd.NA, np.nan: pd.NA})
    df_region = joined.set_axis(pd.MultiIndex.from_product([['Absolute'], cols]), axis=1).join(cumsum).join(pct).join(cagr).join(share).join(share_global)

    
    total_global = df_global[('Absolute', 'Total')]
    total_region = df_region[('Absolute', 'Total')]
    
    
    df_region_total = pd.DataFrame({
        'Absolute count (regional)': df_region['Absolute'].sum(axis=0),
        
        'Share of global total': df_region['Absolute'].sum(axis=0) / df_global[('Absolute', 'Total')].sum() * 100,
        'Share of regional total': df_region['Absolute'].sum(axis=0) / df_region[('Absolute', 'Total')].sum() * 100,
        
        'Share of global total (2006–2015)': df_region.loc[2006:2015]['Absolute'].sum(axis=0) / df_global.loc[2006:2015][('Absolute', 'Total')].sum() * 100,
        'Share of regional total (2006–2015)': df_region.loc[2006:2015]['Absolute'].sum(axis=0) / df_region.loc[2006:2015][('Absolute', 'Total')].sum() * 100,
        
        'Share of global total (2016–2025)': df_region.loc[2016:2025]['Absolute'].sum(axis=0) / df_global.loc[2016:2025][('Absolute', 'Total')].sum() * 100,
        'Share of regional total (2016–2025)': df_region.loc[2016:2025]['Absolute'].sum(axis=0) / df_region.loc[2016:2025][('Absolute', 'Total')].sum() * 100,
        
        'Share of global total (2006–2025)': df_region.loc[2006:2025]['Absolute'].sum(axis=0) / df_global.loc[2006:2025][('Absolute', 'Total')].sum() * 100,
        'Share of regional total (2006–2025)': df_region.loc[2006:2025]['Absolute'].sum(axis=0) / df_region.loc[2006:2025][('Absolute', 'Total')].sum() * 100,
        
       # 'Share of global total, 2016–2025': annual_counts.loc[2016:2025].sum() / df[df['incl_major'] & (df['Publication year']>=2016) & (df['Publication year']<= 2025)].index.nunique() * 100,
       # 'Share of global total, 2006–2015': annual_counts.loc[2006:2015].sum() / df[df['incl_major'] & (df['Publication year']>=2006) & (df['Publication year']<= 2015)].index.nunique() * 100,
       # 
        #'Share of regional total': totals / df[df['incl_major']].index.nunique() * 100,
        #'Share of regional total, 2016–2025': annual_counts.loc[2016:2025].sum() / df[df['incl_major'] & (df['Publication year']>=2016) & (df['Publication year']<= 2025)].index.nunique() * 100,
        #'Share of regional total, 2006–2015': annual_counts.loc[2006:2015].sum() / df[df['incl_major'] & (df['Publication year']>=2006) & (df['Publication year']<= 2015)].index.nunique() * 100,
        
        'Growth (2006–2015)': ((cumsum.loc[2015] - cumsum.loc[2006]) / cumsum.loc[2006] * 100)['CumSum'],
        'Growth (2016–2025)': ((cumsum.loc[2025] - cumsum.loc[2016]) / cumsum.loc[2016] * 100)['CumSum'],
    }).T
    
    display(df_region_total)

    df_region = df_region.set_axis(pd.MultiIndex.from_tuples([(REGION_NAME, *col) for col in df_region.columns], names=['Selection','Metric','Category']), axis=1)
    df_global = df_global.set_axis(pd.MultiIndex.from_tuples([('Global', *col) for col in df_global.columns], names=['Selection','Metric','Category']), axis=1)

    display(Markdown('**Annual metrics**'))
    table = df_region.join(df_global)
    display(table.style.background_gradient(cmap='Blues', axis=0))

    return table

## Primary class (region by location)

In [ ]:
df_counts = annual_counts_info(count_primary_class=True, region_filter='incl_region_loc', label_group='cat', mask=df['incl_major'])

fig, ax = barplot(fname='major_category_loc')
display(fig)
fig.clf()

fig = lineplot(fname='annual_stats_primary_loc')
display(fig)
fig.clf()

## Primary class (region by affiliation)

In [ ]:
df_counts = annual_counts_info(count_primary_class=True, region_filter='incl_region_aff', label_group='cat', mask=df['incl_major'])

fig, ax = barplot(fname='major_category_aff')
display(fig)
fig.clf()

fig = lineplot(fname='annual_stats_primary_aff')
display(fig)
fig.clf()

## Multi-labelling (region by location)

In [ ]:
df_counts = annual_counts_info(count_primary_class=False, region_filter='incl_region_loc', label_group='cat', mask=df['incl_major'])

fig = lineplot(fname='annual_stats_multi_loc')
display(fig)
fig.clf()

## Multi-labelling (region by affiliation)

In [ ]:
df_counts = annual_counts_info(count_primary_class=False, region_filter='incl_region_aff', label_group='cat', mask=df['incl_major'])

fig = lineplot(fname='annual_stats_multi_aff')
display(fig)
fig.clf()

# Geographic distribution

In [ ]:
def plot(mask = None, bins: list[int] | None = None, show_tables: bool = False, show_distributions: bool = False):
    if mask is None:
        mask = np.ones(len(df_locations), dtype=bool)
    
    cnt_adm0 = df_locations[mask].groupby('iso3')['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'})
    cnt_latlon = df_locations[mask & (df_locations['feature_code'] != 'PCLI')].groupby(['LAT', 'LON'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'})
    
    cnt_adm0_yr = df_locations[mask].groupby(['iso3','Publication year'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'}).pivot(index='Publication year', columns='iso3', values='Count').fillna(0).astype(int)
    cnt_latlon_yr = df_locations[mask & (df_locations['feature_code'] != 'PCLI')].groupby(['LAT', 'LON', 'Publication year'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'}).pivot(index='Publication year', columns=['LAT','LON'], values='Count').fillna(0).astype(int)

    if show_tables:
        display(Markdown('### Number of publications per country'))
        display(cnt_adm0.transpose())
        display(cnt_latlon.transpose())
    
        display(Markdown('### Number of publications per country per year'))
        display(cnt_adm0_yr.transpose())
        display(cnt_latlon_yr.transpose())

    if show_distributions:
        display(Markdown('### Distribution of sub-national locations'))
        display(Markdown('\n'.join([
            f'* Percentile {p}: {int(np.percentile(cnt_latlon['Count'], p))}'
            for p in range(90, 101, 1)
        ])))
        fig, axes = plt.subplots(1,4, figsize=(15,3))
        cnt_latlon['Count'].hist(bins=100, ax=axes[0])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 99)]['Count'].hist(bins=50, ax=axes[1])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 98)]['Count'].hist(bins=20, ax=axes[2])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 95)]['Count'].hist(bins=10, ax=axes[3])
        plt.show(fig)
    
    fig = plt.figure(dpi=150)#, figsize=(10, 6.5))
    ax = plt.subplot(projection=AXIS_PROJECTION())
    ax.set_title('Locations of studies on the nexus of climate and health', loc='center', fontsize=10)
    ax.set_extent(EXTENT)
    ax.gridlines()
    
    cmap = mpl.colormaps['BuPu']
    norm = mpl.colors.Normalize(vmin=0, vmax=cnt_adm0['Count'].max())
    
    cnt_adm0.set_index('iso3', inplace=True)
    for _, row in shapes.iterrows():
        iso = row['ISO_A3']
        ax.add_geometries(
            [row['geometry']],
            fc=cmap(norm(cnt_adm0.loc[iso, 'Count'])) if iso in cnt_adm0.index else 'None', 
            crs=PROJECTION(), 
            ec='black', 
            lw=0.2,
        )
    
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=cmap), 
        ax=ax, orientation='horizontal',
        fraction=0.046, pad=0.04
    )
    cbar.set_label('Number of publications mentioning countries')

    
    if bins is not None:
        # Draw bubbles
        norm = mpl.colors.Normalize(vmin=0, vmax=np.percentile(cnt_latlon['Count'], 98))
        ax.scatter(
            cnt_latlon['LON'], cnt_latlon['LAT'], 
            transform=PROJECTION(), 
            s=norm(cnt_latlon['Count']) * 100,
            alpha=0.7,
            color='#ef8a62',
            zorder=10
        )
        
        # Prepare legend for bubbles
        for bin_i in bins: 
            ax.scatter(
                [],[], 
                transform=PROJECTION(), 
                s=norm(bin_i) * 100,
                label=bin_i,
                alpha=0.7,
                color='#ef8a62',
            )

        leg = ax.legend(title='Number of publications\nmentioning sub-national\nlocations', fontsize=8)
        leg.get_title().set_fontsize('8')
    
    fig.tight_layout()
    
    return fig

## Geographic map
These counts include the number of records included in the global indicator that mention a location in Latin America. The analysis has four resolutions: 
* Number of publications per country,
* Number of publications per country and year,
* Number of publications per grid cell and year (2-digits,  [~1km resolution](https://support.oxts.com/hc/en-us/articles/115002885125-Level-of-Resolution-of-Longitude-and-Latitude-Measurements)), and
* Number of publications per grid cell (2-digits,  [~1km resolution](https://support.oxts.com/hc/en-us/articles/115002885125-Level-of-Resolution-of-Longitude-and-Latitude-Measurements)).

The first and last aggregations are plotted.    
For comparison, the distribution and percentiles of counts per grid cell are shown as well.   
The grid cell aggregation is excludion location with 'PCLI' admin codes [(freely associated state)](https://www.geonames.org/export/codes.html)

In [ ]:
fig = plot(show_tables=False, show_distributions=False)

fig.savefig(OUTPUT_DIR / 'geographic_map.png')
fig.savefig(OUTPUT_DIR / 'geographic_map.svg')
fig.savefig(OUTPUT_DIR / 'geographic_map.pdf')

In [ ]:
fig = plot(mask=df_locations['incl_region'], bins=[10, 50, 100], show_tables=False, show_distributions=True)
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.png')
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.svg')
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.pdf')

# Author affiliations vs study locations

In [ ]:
for grouping in [
    #'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    #'Group (HDI 2026)', 
    #'Region (WorldBank 2026)', 
    #'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
    'Name (Lancet 2026)',
]:
    display(Markdown(f'## {grouping}'))
    groups = df_countries[df_countries[grouping].notna() & df_countries['incl_region']][grouping].unique()
    
    subgroups = {
        name: [
            (f'{name} ({len(index):,})', index)
            for name, index in filters
        ]
        
        for name, filters in {
            'Study location': [
                ('None found', df[~df.index.isin(df_locations.index) & df['incl_region_loc']].index.unique()),
                ('Ungrouped', df_locations[df_locations[grouping].isna() & df_locations['incl_region']].index.unique()),
            ] + [
                (group, df_locations[(df_locations[grouping]==group) & df_locations['incl_region']].index.unique())
                for group in groups
            ],
            'Author affiliation': [
                ('Not available', df[~df.index.isin(df_affiliations.index) & df['incl_region_aff']].index.unique()),
                ('Ungrouped', df_affiliations[df_affiliations[grouping].isna() & df_affiliations['incl_region']].index.unique()),
                (f'Not {REGION_NAME}', df[df['incl_region_loc'] & ~df['incl_region_aff']].index.unique()),
            ] + [
                (group, df_affiliations[(df_affiliations[grouping]==group) & df_affiliations['incl_region']].index.unique())
                for group in groups
            ],
        }.items()
    }
    
    data = []
    for name1, index1 in subgroups['Study location']:
        for name2, index2 in subgroups['Author affiliation']:
            data.append({
                'Weight': len(index1.intersection(index2)),
                'Study location': name1,
                'Author affiliation': name2,
            })
    sdf = pd.DataFrame(data)
    fig = plt.figure(figsize=(7.4, 7), dpi=150)
    sankey(sdf[sdf['Weight']>0], cmap=plt.get_cmap('viridis'), labels_size=8, titles_size=10)
    plt.tight_layout()
    display(fig)

    
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.png')
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.svg')
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.pdf')
    
    fig.clf()
    fig.clear()

# Regions

In [ ]:
def produce_regional_counts(grouping: str, df_geo: pd.DataFrame, prefix: str):
    display(Markdown(f'### {grouping}'))
    regions = df_geo[grouping].unique()
    
    mask = df['incl_major'] & df['incl_location']# & df['incl_region_loc']
    
    buffer = []
    buffer_totals = []
    for region in regions:
        if type(region) is not str:
            continue
        logger.info(f'Counting for {region}')
        annual_counts, totals, annual_totals = label_group_counts(
            df[mask & df.index.isin(df_geo[df_geo[grouping] == region].index)], 
            group=LABELS['cat'],
            geography_filter='location',
            count_primary_class=True,
        )
        annual_counts['Total'] = annual_totals
        annual_counts.columns = pd.MultiIndex.from_product([[region], annual_counts.columns])
        totals['Total'] = annual_totals.sum()
        totals.index = pd.MultiIndex.from_product([[region], totals.index])
        buffer_totals.append(totals)
        buffer.append(annual_counts)
    regional_counts = pd.concat(buffer, axis=1)
    regional_totals = pd.concat(buffer_totals)
    annual_total = df[mask].groupby('Publication year')['item_id'].nunique()

    annual_group_population = df_population.merge(df_countries, left_on='iso3', right_on='iso3', how='left').groupby([grouping, 'year'])['Population'].sum().reset_index().pivot(index='year', values='Population', columns=grouping)
    annual_group_population.loc[2025] = annual_group_population.loc[2024]  # No data for 2025 yet
    
    prc_pop = regional_counts / annual_group_population.rename_axis(index='Publication year', columns=None) * 100000
    prc_cagr = ((((regional_counts.pct_change(5) + 1) ** (1/5)) - 1) * 100).iloc[5:].replace({np.inf:pd.NA, np.nan: pd.NA})
    prc_pct = regional_counts.pct_change().replace({np.inf:pd.NA, np.nan: pd.NA}) * 100
    prc_per = ((regional_counts.T / regional_counts.T.xs('Total', level=1)).T*100).replace({np.inf:pd.NA, np.nan: pd.NA})
    prc_sha = (regional_counts.T / annual_total*100).replace({np.inf:pd.NA, np.nan: pd.NA}).T
    
    mask = df['incl_major'] & df['incl_location'] & df.index.isin(df_geo[df_geo[grouping].isin(regions)].index)

    display(Markdown(f'#### Totals\nNumber of unique records: {df[mask].index.nunique():,}'))
    display(pd.DataFrame({
        'Total (abs)': regional_totals.astype(int),
        'Total (share per group)':regional_totals/ regional_totals.xs('Total', level=1) * 100,
        'Total (share of total)': regional_totals / df[mask].index.nunique() * 100,
        'Total (share of total, 2021–2025)': regional_counts.loc[2021:2025].sum() / df[mask & (df['Publication year']>=2021) & (df['Publication year']<= 2025)].index.nunique() * 100,
        'Total (share of total, 2016–2020)': regional_counts.loc[2016:2020].sum() / df[mask & (df['Publication year']>=2016) & (df['Publication year']<= 2020)].index.nunique() * 100,
        'Total (per 100k citizens)': prc_pop.mean(axis=0),
        'Total (per 100k citizens, 2021–2025)': prc_pop.loc[2021:2025].mean(axis=0),
        'Total (per 100k citizens, 2016–2020)': prc_pop.loc[2016:2020].mean(axis=0),
    }).T)

    fig, axes = plt.subplots(1, 5, figsize=(25, 2.5), dpi=150)

    for ai, (sub, title) in enumerate([
        (regional_counts, 'Annual counts'),
        (prc_sha, 'Annual shares'),
        (prc_pct, 'Annual growth'),
        (prc_cagr, 'CAGR @ 5yrs'),
        (prc_pop, 'Records per 100k citizens'),
    ]):
        tab = sub.T.xs('Total', level=1).T.fillna(np.nan)
        for col in tab.columns:
            axes[ai].plot(tab.index, tab[col], label=col, lw=1)
            axes[ai].set_title(title)
    axes[-1].legend(bbox_to_anchor=(1.1, 1.05))

    display(fig)
    
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.png')
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.svg')
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.pdf')
    
    fig.clf()
    fig.clear()

    display(Markdown('#### Annual counts'))
    display(regional_counts.astype('Int64'))

    display(Markdown('#### Annual shares per group'))
    display(prc_per)
    
    display(Markdown('#### Annual shares of all'))
    display(prc_sha)

    display(Markdown('#### Annual growth'))
    display(prc_pct)
    
    display(Markdown('#### CAGR @ 5yrs'))
    display(prc_cagr)

    display(Markdown('#### Annual counts by population\nNumbers are normalised to records per 100,000 citizens'))
    display(prc_pop)

## Regions by location

In [ ]:
for grp in [
    'Name (Lancet 2026)', 
    #'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    #'Group (HDI 2026)', 
    #'Region (WorldBank 2026)', 
    #'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)',
    #'Region (EEA sub-division)',
    #'Region (UN Europe sub-region)',
    #'EU member (2025)',
    #'EEA member (2025)',
]:
    produce_regional_counts(grp, df_geo=df_locations[df_locations['incl_region']], prefix='locations')

## Regional counts by affiliation

In [ ]:
for grp in [
    'Name (Lancet 2026)', 
    #'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    #'Group (HDI 2026)', 
    #'Region (WorldBank 2026)', 
    #'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
    #'Region (EEA sub-division)',
    #'Region (UN Europe sub-region)',
    #'EU member (2025)',
    #'EEA member (2025)',
]:
    produce_regional_counts(grp, df_geo=df_affiliations[df_affiliations['incl_region']], prefix='affiliations')

# Climate impacts

Think of the following counts as coming from a chain of filters. Where applicable, the appropriate shares of reference totals is available
* relevant for climate and health (incl_major)
* Categorised as 'impacts' ('cat|2'>0.5)
* relevant for climate impacts with dedicated inclusion criterion (incl_impacts)
* location mentioned in abstract (incl_location)
* location is in region (incl_region_loc)
* mentioned location is in or intersects with grid cell with attributable climate change

In [ ]:
df_imp = pd.DataFrame({
    'Relevant records': df[df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant with location': df[df['incl_location'] & df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant records and category impacts': df[df['incl_major'] & (df['cat|2']>0.5)].groupby('Publication year')['item_id'].nunique(),
    'Relevant on impacts': df[df['incl_impacts'] & df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant on impacts with location': df[df['incl_location'] & df['incl_major'] & df['incl_impacts']].groupby('Publication year')['item_id'].nunique(),
    'Records on impacts with location in attributable cell':  df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
    'Records on impacts with location in non-attributable cell': df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & ~df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
    
    f'Relevant on impacts with location in {REGION_NAME}': df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['incl_region_loc']].groupby('Publication year')['item_id'].nunique(),
    f'Records on impacts with location in {REGION_NAME} in attributable cell':  df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['incl_region_loc'] & df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
    f'Records on impacts with location in {REGION_NAME} in non-attributable cell': df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['incl_region_loc'] & ~df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
})
display(Markdown('## Totals'))
display(pd.DataFrame({
    '2006–2015': df_imp.loc[2006:2015].sum(),
    '2016–2025': df_imp.loc[2016:2025].sum(),
}).T.astype('Int32'))

display(Markdown('## Anual counts'))
display(pd.concat([df_imp, df_imp.sum().rename('Total').to_frame().T]).astype('Int32'))

display(Markdown('## Cummulative counts'))
display(df_imp.cumsum())

display(Markdown('## Annual growth'))
display(df_imp.cumsum().pct_change()*100)

## Label counts (region based on study location)

In [ ]:
pd.options.display.max_columns = 650
pd.options.display.max_rows = 40

for key in [
    'driver', 
    'event', 
    'health', 
    'expose',
    'attr',
    #'sector', 
    'keywords',
   #'topic-agg',
    'topic-agg-agg',
]:
    group = LABELS[key]

    display(Markdown(f'### {group.name}'))
    
    _ = annual_counts_info(count_primary_class=False, region_filter='incl_region_loc', label_group=key, mask=df['incl_impacts'] & df['incl_major'] & df['incl_location'] & df['grid_attributable'])

## Label counts (region based on author affiliation)

In [ ]:
pd.options.display.max_columns = 650
pd.options.display.max_rows = 40

for key in [
    'driver', 
    'event', 
    'health', 
    'expose',
    'attr',
   # 'sector', 
    'keywords',
   #'topic-agg',
    'topic-agg-agg',
]:
    group = LABELS[key]

    display(Markdown(f'### {group.name}'))
    
    _ = annual_counts_info(count_primary_class=False, region_filter='incl_region_aff', label_group=key, mask=df['incl_impacts'] & df['incl_major'] & df['incl_location'] & df['grid_attributable'])

## Attributions on geographic map

In [ ]:
lim_percentile = (0, np.percentile(grid_counts[grid_counts['studies_per_capita'].notna()]['studies_per_capita'], 0.99))
lim_median = (0, grid_counts[grid_counts['studies_per_capita'].notna()]['studies_per_capita'].median() * 50)
print(f'Upper 95th percentile limit for per capita: {lim_percentile}')
print(f'50x median limit for per capita: {lim_median}')

grid_counts.describe()

### Population / number of studies plots
Figure includes *all* impacts studies!
Overlay shows where we have attributable effects (`//` for OR, `xx` for AND)

In [ ]:
#mpl.rcParams['hatch.linewidth'] = 0.1  # previous pdf hatch linewidth
#mpl.rcParams['hatch.linewidth'] = 1.0  # previous svg hatch linewidth
mpl.rcParams['hatch.linewidth'] = 0.2

for attribution_layer in ['', '_with_attribution_land', '_with_attribution_all']:
    fig, axes = plt.subplots(1,3, dpi=150, figsize=(16, 5), subplot_kw={'projection': ccrs.EqualEarth()})
    for ax, (field, name, scaling, lim, norm) in zip(axes, [
        ('population', 'Number of citizens', 1, None, True),
        ('study_count', 'Number of publications', 1, None, True),
        ('studies_per_capita', 'Publications per 1,000,000 citizens', 1000000, lim_median, True),
    ]):
        grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values=field).astype('float32').replace({pd.NA: np.nan}).fillna(np.nan)
        if lim:
            grid = grid.clip(*lim)
        mesh = ax.pcolormesh(
            # LON, LAT, DATA
            grid.columns, grid.index, grid.values * scaling,
            cmap=mpl.colormaps.get_cmap('Greens'),
            norm=mpl.colors.LogNorm() if norm else None,
            transform=ccrs.PlateCarree(),
        )

        if attribution_layer != '':
            if attribution_layer == '_with_attribution_land':
                grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values='attribution')
            else: # _with_attribution_all
                grid = pd.pivot_table(grid_counts, index='LAT', columns='LON', values='attribution')

            patterns = ax.contourf(
                grid.columns, grid.index, grid.values,
                levels=[-0.5, 0.5, 1.5, 2.5], 
                # '': transparent, '.': dotted, '\\': lines
                hatches=['', '/////', 'xxxxxxx'],
                colors='none', # Makes the background of the hatch transparent
                transform=ccrs.PlateCarree(),
                corner_mask=False,
            )
        
        ax.set_extent(EXTENT)
        ax.coastlines(lw=0.2)
        
        fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05, label=name)
    
    fig.tight_layout()
    
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.png')
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.svg')
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.pdf')

    display(fig)
    
    fig.clf()

### Per capita and number of study plots
Because we have two types of locations, countries (etc) and cities (etc), we need the figure above has two sources of the counts per grid cell.
This plot shows both sources individually.

In [ ]:
fig, axes = plt.subplots(2,2, dpi=150, figsize=(16,10), subplot_kw={'projection': ccrs.EqualEarth()})

for ax, field, name, scaling, lim, norm in [
    (axes[0][0], 'study_count_abstracted', 'Number of publications (country-level etc)', 1, None, True),
    (axes[0][1], 'studies_per_capita_abstracted', 'Publications per 1,000,000 citizens (country-level etc)', 1000000, None, True),
    (axes[1][0], 'study_count_direct', 'Number of publications (regional etc)', 1, None, True),
    (axes[1][1], 'studies_per_capita_direct', 'Publications per 1,000,000 citizens (regional etc)', 1000000, lim_median, True),
]:
    grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values=field).astype('float32').replace({pd.NA: np.nan}).fillna(np.nan)
    if lim:
        grid = grid.clip(*lim)
    mesh = ax.pcolormesh(
        # LON, LAT, DATA
        grid.columns, grid.index, grid.values * scaling,
        cmap=mpl.colormaps.get_cmap('Greens'),
        norm=mpl.colors.LogNorm() if norm else None,
        transform=ccrs.PlateCarree(),
    )
    ax.set_extent(EXTENT)
    
    ax.coastlines(lw=0.2)
    
    fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05, label=name)
fig.tight_layout()

fig.savefig(OUTPUT_DIR / 'global_distribution_split.png')
fig.savefig(OUTPUT_DIR / 'global_distribution_split.svg')
fig.savefig(OUTPUT_DIR / 'global_distribution_split.pdf')

### Attribution grid

In [ ]:
tmp = df_grid.copy()
tmp['temp'] = 1
tmp.loc[tmp['grid_cooler'], 'temp'] = 0
tmp.loc[tmp['grid_warmer'], 'temp'] = 2

tmp['prec'] = 1
tmp.loc[tmp['grid_drier'], 'prec'] = 0
tmp.loc[tmp['grid_wetter'], 'prec'] = 2

fig, axes = plt.subplots(1,3, dpi=150, figsize=(16,7), subplot_kw={'projection': ccrs.EqualEarth(central_longitude=0)})

axes[0].set_title('a) Attributable trends in temperature')
axes[0].coastlines(lw=0.2)
grid = pd.pivot_table(tmp.mask(~tmp['is_land']), index='LAT', columns='LON', values='temp')
axes[0].pcolormesh(grid.columns, grid.index, grid.values * 0 + 1, cmap=mpl.colors.ListedColormap(["#000000"]), transform=ccrs.PlateCarree())
mesh = axes[0].pcolormesh(grid.columns, grid.index, grid.values, vmin=0, vmax=2, cmap=mpl.colors.ListedColormap(['#67a9cf', '#f1f1f1', '#ef8a62']), transform=ccrs.PlateCarree())
bar = fig.colorbar(mesh, ax=axes[0], location='bottom', shrink=0.75, pad=0.05)
bar.set_ticks(ticks=[0,1,2], labels=['Cooler', 'Not attributable', 'Warmer'], fontsize=9)


axes[1].set_title('b) Attributable trends in precipitation')
axes[1].coastlines(lw=0.2)
grid = pd.pivot_table(tmp.mask(~tmp['is_land']), index='LAT', columns='LON', values='prec')
axes[1].pcolormesh(grid.columns, grid.index, grid.values * 0 + 1, cmap=mpl.colors.ListedColormap(["#000000"]), transform=ccrs.PlateCarree())
mesh = axes[1].pcolormesh(grid.columns, grid.index, grid.values, vmin=0, vmax=2, cmap=mpl.colors.ListedColormap(['#ef8a62', '#f1f1f1', '#67a9cf']), transform=ccrs.PlateCarree())
bar = fig.colorbar(mesh, ax=axes[1], location='bottom', shrink=0.75, pad=0.05)
bar.set_ticks(ticks=[0,1,2], labels=['Drier', 'Not attributable', 'Wetter'], fontsize=9)


axes[2].set_title('c) Density of impact studies')
axes[2].coastlines(lw=0.2)
grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values='studies_per_capita').astype('float32').replace({pd.NA: np.nan}).fillna(np.nan).clip(*lim_median) * 1000000
mesh = axes[2].pcolormesh(grid.columns, grid.index, grid.values, cmap=mpl.colormaps.get_cmap('Greens'), norm=mpl.colors.LogNorm(), transform=ccrs.PlateCarree())
fig.colorbar(mesh, ax=axes[2], location='bottom', shrink=0.75, pad=0.05, label='Publications per 1,000,000 citizens')

axes[0].set_extent(EXTENT)
axes[1].set_extent(EXTENT)
axes[2].set_extent(EXTENT)

fig.tight_layout()
plt.show(fig)

fig.savefig(OUTPUT_DIR / 'attribution_horizontal.png')
fig.savefig(OUTPUT_DIR / 'attribution_horizontal.svg')
fig.savefig(OUTPUT_DIR / 'attribution_horizontal.pdf')

## Exposed people

In [ ]:
for grp in [
    'Name (Lancet 2026)',
    #'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    #'Group (HDI 2026)', 
    #'Group (HDI-2 2026)', 
    #'Region (WorldBank 2026)', 
    #'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
]:
    display(Markdown(f'### {grp}'))
    df_exposed = pd.DataFrame({
        'Population (in M)': df_countries[df_countries['incl_region']].groupby(grp)['Population (Lancet, 2025)'].sum() / 1000000,
        'Number of studies': df_locations[df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location'] & df_locations['grid_attributable'] & df_locations['incl_region']].groupby(grp)['item_id'].nunique(),
    }).drop('French Guiana')
    df_exposed['Studies per million citizens exposed'] = df_exposed['Number of studies']  / df_exposed['Population (in M)']

    display(df_exposed)
    

    fig, ax = plt.subplots(1, 1, figsize=(10,8), dpi=150)
    axes=[ax]
    W = 0.35
    h1 = axes[0].barh(np.arange(len(df_exposed.index))+(W/2), df_exposed['Number of studies'], height=W, label='Number of studies', facecolor='#F768A0')
    h2 = axes[0].barh(np.arange(len(df_exposed.index))-(W/2), df_exposed['Population (in M)'], height=W, label='Population in million', facecolor='#AD017D')
    axes[0].set_yticks(range(df_exposed.shape[0]))
    axes[0].set_yticklabels(df_exposed.index)
    

    ax = axes[0].twiny()
    h3 = ax.barh(df_exposed.index, df_exposed['Studies per million citizens exposed'], label='Studies per million citizens', height=0.15, facecolor='#FAB4B8', hatch='\\')
    ax.set_ylabel('Studies per million citizens')
    #fig.legend(loc='upper right')
    axes[0].legend(handles=[h1,h2,h3])
    fig.tight_layout()
    
    #axes[1].set_title('Studies per million citizens exposed', fontsize=10)
    #axes[1].barh(df_exposed.index, df_exposed['Studies per million citizens exposed'], label='Studies per million citizens', height=0.5)
    #axes[1].set(yticklabels=[])
    display(fig)
    
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.png')
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.pdf')
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.svg')
    fig.clf()
    fig.clear()

## Sankeys

In [ ]:
df.columns

In [ ]:
ATTRIBUTABLE = {
    'driver|1': ('Changes in temperature', 'temp_da'),
    #'climateDrivers|2': 'Seasonal Change',
    'driver|3': ('Changes in precipitation', 'precip_da'),
    #'climateDrivers|4': 'Sea-level rise',
    #'climateDrivers|5': 'Climate Change (unspecified) ',
    #'climateDrivers|7': 'Other meteorological variables',
    #'climateDrivers|6': 'Changes in humidity'},
}

mask_base = df['incl_major'] & df['incl_impacts'] & df['incl_location'] & df['incl_region_loc']
mask_attr = df['grid_attributable'] == True

lab1 = LABELS['driver']
lab2 = LABELS['health']

n_a = df[mask_base & mask_attr]['item_id'].nunique()
n_na = df[mask_base & ~mask_attr]['item_id'].nunique()

data = []
for i, label1 in enumerate(lab1.labels):
    name1 = label1.name
    key1 = label1.column
    if key1 not in df: continue
    print(name1, key1)
    m1 = (df[key1] >= 0.5) 
    n1 = df[m1 & mask_base]['openalex_id'].nunique()

    for j, label2 in enumerate(lab2.labels):
        name2 = label2.name
        key2 = label2.column
        if key2 not in df: continue
        m2 = (df[key2] >= 0.5)
        n2 = df[m2 & mask_base]['item_id'].nunique()
        
        if key1 in ATTRIBUTABLE:
            data.append([
                df[mask_base & mask_attr & m1 & m2]['item_id'].nunique(),
                f'Attributable ({n_a:,})',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])
            data.append([
                df[mask_base & ~mask_attr & m1 & m2]['item_id'].nunique(),
                f'Not attributable ({n_na:,})',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])
        else:
            data.append([
                df[mask_base & m1 & m2]['item_id'].nunique(),
                'NA',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])

sdf = pd.DataFrame(data, columns=['Weight', 'In area where trend is', lab1.name, lab2.name])

fig = plt.figure(figsize=(7.4, 7), dpi=150)
sankey(sdf, cmap=plt.get_cmap('viridis'), labels_size=8, titles_size=10)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / 'sankey.png')
fig.savefig(OUTPUT_DIR / 'sankey.svg')
fig.savefig(OUTPUT_DIR / 'sankey.pdf')